# Distil the student, then serve it with vLLM

**Company server, PRIVATE data.** Fine-tune a small student (Qwen3-VL-8B, the
served base) on the teacher's corrected labels — sequence-level distillation — then
hot-load the adapter into vLLM and A/B it against the base live. All local.

## 1. Config + load the teacher labels

In [ ]:
import sys, json; sys.path.insert(0, "..")
from pathlib import Path

from src.data.difficulty import score_table
from src.data.loader import TableRecord
from src.model.registry import MODEL_8B
from src.ocr.engine import run_ocr
from src.ocr.layout import serialize_layout
from src.train.lora import TrainConfig, train
from src.model.prompts import build_schema_instruction

STUDENT = MODEL_8B                     # must equal the vLLM-served base checkpoint
LABELS = Path("data/invoices/teacher_labels.jsonl")
INSTRUCTION = build_schema_instruction()   # MUST match the teacher / inference prompt

rows = [json.loads(l) for l in LABELS.read_text().splitlines() if l.strip()]
print(f"{len(rows)} corrected labels")

# Distillation: teacher HTML is the training target (passed via `targets`). Ground
# with the same stage-1 layout the teacher saw, so train/infer prompts match.
records, targets, layouts = [], {}, {}
for r in rows:
    uid, img, html = r["uid"], r["image"], r["html"]
    records.append(TableRecord(uid=uid, image_path=img, html="", complexity=score_table(html)))
    targets[uid] = html
    layouts[uid] = serialize_layout(run_ocr(img), style="grid")

## 2. Train the LoRA adapter (schema mode, grounded, distilled)

In [ ]:
cfg = TrainConfig(model_id=STUDENT, mode="schema", instruction=INSTRUCTION,
                  output_dir="outputs/lora", epochs=3)
model, processor, trainer = train(records, cfg, ocr_layouts=layouts, targets=targets)
ADAPTER = Path(cfg.output_dir) / "adapter"
print("adapter ->", ADAPTER)

## 3. Serve base + adapter with vLLM (run once, in a terminal)

```bash
vllm serve Qwen/Qwen3-VL-8B-Instruct --enable-lora \
    --lora-modules invoice=outputs/lora/adapter --port 8001
```
`--enable-lora` hot-loads the adapter with no weight merge, so base-vs-adapter A/B
is against identical base weights.

## 4. Live A/B — base vs distilled adapter, on a held-out invoice

In [ ]:
from src.model.vllm_client import VLLMTableReconstructor

URL = "http://localhost:8001/v1"
base    = VLLMTableReconstructor(model_id="Qwen/Qwen3-VL-8B-Instruct", base_url=URL)
adapter = VLLMTableReconstructor(model_id="invoice", base_url=URL)   # the --lora-modules name

img = records[-1].image_path
layout = layouts[records[-1].uid]
b = base.predict(img, instruction=INSTRUCTION, ocr_layout=layout)
a = adapter.predict(img, instruction=INSTRUCTION, ocr_layout=layout)

from IPython.display import HTML, display
print("=== base ==="); display(HTML(b.html or "<i>empty</i>"))
print("=== adapter ==="); display(HTML(a.html or "<i>empty</i>"))

---
Once ~20 corrected labels + synthetic invoices reach trainable volume, this same
cell trains on the larger set. Adapters must target the exact served checkpoint or
they will not load.